John S. Anvik, 2025  
University of Lethbridge Student ID: 001224816

### Import Packages

In [158]:
# Import packages

library(tidyverse)

### Load and Trim Dataset

In [159]:
# minizord.csv already removed non-songs, post-dawn, unmeasureable songs, acoustic outliers
# see "Data Preprocessing/01 preprocessing.ipynb"
data <- read_csv("../data/minizord.csv")

# Only look at North data for now
data <- data %>% filter(neighbourhood == "north") %>% ungroup()

Rows: 7594 Columns: 52
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr   (4): ID, IST, neighbourhood, Location_Number
dbl  (45): time_rel_sunrise, UST, D_M1_M2, D_M1_M3, D_M1_M4, D_M1_M5, D_M1_M...
lgl   (1): RepeatST
dttm  (1): DateTime
date  (1): date

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


### Reformat Data: Row for each possible matching opportunity of focal male

In [160]:
# Just focusing on one focal male for now (central male LbOW)
focal_id <- "LbOW"
focal_match_data <- data

In [161]:
ids <- c("LbOW", "RbDgY", "YKY", "TbTbLg", "DgTbO", "RRO") # Ordered list of IDs correstponding to M1..M6
for (i in 1:6) {
    print(ids[[i]])
    for (d in unique(na.omit(filter(data, ID==ids[[i]])$date))) {
        print(d)
        print(nrow(filter(data, ID == ids[[i]], date == d)))
    }
}

[1] "LbOW"
[1] 19125
[1] 5
[1] 19126
[1] 8
[1] 19129
[1] 126
[1] 19130
[1] 106
[1] "RbDgY"
[1] 19128
[1] 180
[1] 19129
[1] 191
[1] 19130
[1] 191
[1] "YKY"
[1] 19125
[1] 128
[1] 19126
[1] 131
[1] 19128
[1] 247
[1] 19129
[1] 214
[1] 19130
[1] 199
[1] "TbTbLg"
[1] 19125
[1] 141
[1] 19126
[1] 131
[1] 19128
[1] 127
[1] 19129
[1] 63
[1] 19130
[1] 59
[1] "DgTbO"
[1] 19125
[1] 58
[1] 19126
[1] 147
[1] 19128
[1] 153
[1] 19129
[1] 123
[1] 19130
[1] 163
[1] "RRO"
[1] 19125
[1] 14
[1] 19126
[1] 82
[1] 19128
[1] 46
[1] 19129
[1] 62
[1] 19130
[1] 30


In [162]:
print(unique(na.omit(data$ID)))

[1] "DgTbO"  "LbOW"   "RRO"    "RbDgY"  "TbTbLg" "YKY"   


In [163]:
# Make column for time since Male N's last song
# NA values at top of each day until that male sings
ids <- c("LbOW", "RbDgY", "YKY", "TbTbLg", "DgTbO", "RRO") # Ordered list of IDs correstponding to M1..M6
for (i in 1:6) {
    print(ids[[i]])
    time_last_song <- sprintf("time_M%i_last_song", i)
    time_since_last_song <- sprintf("time_since_M%i_last_song", i)
    print(time_last_song)
    focal_match_data <- focal_match_data %>%
    arrange(DateTime) %>%
    mutate("{time_last_song}" := if_else(ID == ids[[i]], time_rel_sunrise, NA)) %>%
    group_by(date) %>%
    fill({time_last_song}, .direction = "down") %>%
    mutate("{time_since_last_song}" := time_rel_sunrise - !!sym({time_last_song})) %>%
    ungroup()
}

[1] "LbOW"
[1] "time_M1_last_song"
[1] "RbDgY"
[1] "time_M2_last_song"
[1] "YKY"
[1] "time_M3_last_song"
[1] "TbTbLg"
[1] "time_M4_last_song"
[1] "DgTbO"
[1] "time_M5_last_song"
[1] "RRO"
[1] "time_M6_last_song"


In [164]:
slice_sample(focal_match_data, n=5)

ID,DateTime,date,time_rel_sunrise,UST,IST,D_M1_M2,D_M1_M3,D_M1_M4,D_M1_M5,⋯,time_M2_last_song,time_since_M2_last_song,time_M3_last_song,time_since_M3_last_song,time_M4_last_song,time_since_M4_last_song,time_M5_last_song,time_since_M5_last_song,time_M6_last_song,time_since_M6_last_song
<chr>,<dttm>,<date>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
RRO,2022-05-14 05:43:51,2022-05-14,-728.0436,30,RRO_11,NA,56.11669,67.67739,62.95599,⋯,NA,NA,-850.8153,122.771726,-731.7632,3.71966,-738.5271,10.483525,-728.0436,0.000000
TbTbLg,2022-05-14 05:33:51,2022-05-14,-1328.6690,17,TbTbLg_8,NA,52.93973,67.67739,62.95599,⋯,NA,NA,-1335.7519,7.082905,-1328.6690,0.00000,-1330.3362,1.667175,-1334.0519,5.382876
DgTbO,2022-05-18 05:21:14,2022-05-18,-2025.5553,6,DgTbO_19,78.37111,47.44059,NA,62.95599,⋯,-2114.4945,88.939204,-2033.3708,7.815522,NA,NA,-2025.5553,0.000000,NA,NA
LbOW,2022-05-17 05:41:44,2022-05-17,-795.4521,17,LbOW_11,94.03680,39.73681,67.67739,60.45910,⋯,-802.3463,6.894188,-804.5254,9.073261,-930.1219,134.66978,-829.2800,33.827911,-930.0887,134.636539
LbOW,2022-05-18 05:26:36,2022-05-18,-1703.4000,29,LbOW_7,94.03680,47.44059,61.21936,62.95599,⋯,-1707.0627,3.662756,-1708.7871,5.387110,-1840.8214,137.42139,-1706.1289,2.728978,-1848.1004,144.700410


In [165]:
# Focal Male's song-type repertoire
focal_repertoire <- unique(filter(data, ID == focal_id)$UST)
focal_repertoire

# Select only rows with focal male's songs
focal_match_data <- focal_match_data %>%
    filter(ID == focal_id) %>%
# Input variables: M2-M6's last UST, time since M1-M6 UST, # M2-M6's distance
    select(
        "D_M1_M2", "D_M1_M3", "D_M1_M4", "D_M1_M5", "D_M1_M6",
        UST,
        # TODO: Add M1 last UST, but shift it 1 back (not M1's current song, but M1's previous song)
        "M2_last_UST", "M3_last_UST", "M4_last_UST", "M5_last_UST", "M6_last_UST",
        # time_M1_last_song, time_M2_last_song, time_M3_last_song, time_M4_last_song, time_M5_last_song, time_M6_last_song,
    	time_since_M2_last_song, time_since_M3_last_song, time_since_M4_last_song, time_since_M5_last_song, time_since_M6_last_song, # did not include time since M1 last song because it is always 0
        number_of_matches, does_it_match,
        DateTime, time_rel_sunrise
        # Also consider adding "streak"
        # Also consider adding MX nth last song
    ) %>% 
    mutate(across(contains("UST"), ~ as.factor(.x)), 
        does_it_match = as.factor(does_it_match),
        number_of_matches = as.integer(number_of_matches)) %>%
# Set all neighbours' songs that aren't in M1's repertoire to 0
    mutate(across(contains("UST"), ~ if_else (.x %in% focal_repertoire, .x, as.character(0))))

[1]  10   5   8  13 120  29   3   9  17  21  NA 117 119

In [166]:
print(1)
print(focal_repertoire)
for (i in 2:6) {
    print(i)
    print(pull(unique(focal_match_data %>% select({sprintf("M%i_last_UST", i)}))))
}


[1] 1
 [1]  10   5   8  13 120  29   3   9  17  21  NA 117 119
[1] 2
[1] NA   "9"  "8"  "21" "13" "0"  "5"  "3"  "17"
[1] 3
[1] "0"  "3"  "10" "9"  "8"  "13" "21" "5"  "17"
[1] 4
 [1] "3"  "10" "0"  "5"  "8"  NA   "9"  "13" "21" "17"
[1] 5
[1] "10" "0"  "8" 
[1] 6
[1] "0"  "9"  "13" "10" "5"  NA   "8"  "3" 


In [167]:
focal_match_data

D_M1_M2,D_M1_M3,D_M1_M4,D_M1_M5,D_M1_M6,UST,M2_last_UST,M3_last_UST,M4_last_UST,M5_last_UST,M6_last_UST,time_since_M2_last_song,time_since_M3_last_song,time_since_M4_last_song,time_since_M5_last_song,time_since_M6_last_song,number_of_matches,does_it_match,DateTime,time_rel_sunrise
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<fct>,<dttm>,<dbl>
NA,46.47336,67.67739,62.95599,53.63760,10,NA,0,3,10,0,NA,16.627895,5.9378387,17.6298851,260.894578,1,1,2022-05-13 05:38:15,-1094.3959
NA,53.50176,67.67739,62.95599,53.63760,5,NA,0,10,10,0,NA,22.562286,4.8127364,23.5642759,266.828969,0,0,2022-05-13 05:38:21,-1088.4615
NA,50.01110,67.67739,62.95599,53.63760,8,NA,0,0,0,0,NA,39.138035,2.0612689,6.1525259,305.756580,0,0,2022-05-13 05:39:00,-1049.5339
NA,46.47336,67.67739,62.95599,53.63760,10,NA,0,0,10,9,NA,15.955015,8.7521505,65.9993951,490.559032,1,1,2022-05-13 05:42:05,-864.7315
NA,54.16852,67.67739,62.95599,53.63760,13,NA,0,5,10,13,NA,46.918660,16.8974710,49.0656891,955.315532,1,1,2022-05-13 05:49:50,-399.9750
NA,52.93973,NA,62.95599,91.08596,120,NA,3,8,0,10,NA,22.855711,NA,0.6582250,NA,0,0,2022-05-14 05:22:34,-2005.6074
NA,52.93973,62.79695,62.95599,91.08596,10,NA,0,0,0,13,NA,11.978942,27.0370271,10.9187442,NA,0,0,2022-05-14 05:23:24,-1955.0779
NA,52.93973,67.67739,62.95599,104.82346,10,NA,0,0,0,10,NA,1.440189,86.5330145,0.7802002,30.500925,1,1,2022-05-14 05:24:24,-1895.5820
NA,52.93973,61.21936,62.95599,53.63760,29,NA,0,0,0,0,NA,8.887690,10.3566827,0.4265619,212.807201,0,0,2022-05-14 05:27:26,-1713.2757


### Export Data

In [168]:
write_csv(focal_match_data, "../data/matching_opportunities.csv")